# Lab Day 1 — MSSV 2A202602853

Bài hoàn chỉnh Part 0–4: M-base, baseline 3 seed, 7 chủ đề thí nghiệm, chọn bằng validation và đánh giá cuối.
Chạy từ repo hoặc thư mục bài nộp/code cùng data/, scripts/ và templates/ của repo.
Runtime khác kết quả đã khóa ghi vào reproduction_<hash>. LAB_OUT_DIR có thể là đường dẫn tuyệt đối tới Drive/Kaggle.
JSON chỉ lưu metric; pipeline huấn luyện lại lấy best-validation weights nếu cần đánh giá mới.


In [1]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np
import torch

# Chạy từ repo hoặc một thư mục con của repo; trên Colab cd vào repo trước.
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "data/covtype.csv.gz").is_file()
                  and (p / "scripts/split_data.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Hãy chuyển working directory vào repo trước khi chạy.")
CODE_DIR = REPO_ROOT / "submission_2A202602853/code"
OUT_DIR = os.environ.get("LAB_OUT_DIR", str(CODE_DIR.parent))
if not Path(OUT_DIR).is_absolute():
    raise ValueError("LAB_OUT_DIR phải là đường dẫn tuyệt đối")
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
REPO_ROOT = str(REPO_ROOT)

torch.set_num_threads(min(2, os.cpu_count() or 1))
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "Không có CUDA; chạy CPU")
print("REPO_ROOT:", REPO_ROOT)
print("OUT_DIR:", OUT_DIR)
for folder in ("figures", "results"):
    Path(OUT_DIR, folder).mkdir(parents=True, exist_ok=True)

from data import prepare_data, load_split, make_val_split, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx
set_seed(42)


PyTorch: 2.8.0+cu128
Device: cpu
GPU: Không có CUDA; chạy CPU
REPO_ROOT: C:\Su\VinLab\phase2-track4\Lec1-Noron\K4-Track4-Day1-Neuralnetwork
OUT_DIR: C:\Su\VinLab\phase2-track4\Lec1-Noron\K4-Track4-Day1-Neuralnetwork\submission_2A202602853


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [2]:
# ===== Part 0: chia dữ liệu, chuẩn hoá và kiểm tra =====
completed = subprocess.run(
    [sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT,
    check=True, capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONUTF8": "1"},
)
print(completed.stdout)
data = prepare_data(device, val_fraction=0.2, seed=42,
                    processed_dir=f"{REPO_ROOT}/data/processed")

expected = {"tr": 371_847, "val": 92_962, "eval": 116_203}
ratios = []
for name, n in expected.items():
    X, y = data[f"X_{name}"], data[f"y_{name}"]
    assert X.shape == (n, 54) and X.dtype == torch.float32
    assert y.shape == (n,) and y.dtype == torch.int64
    assert X.device.type == device and y.device.type == device
    assert torch.isfinite(X).all().item()
    assert ((y >= 0) & (y < 7)).all().item()
    assert ((X[:, 10:] == 0) | (X[:, 10:] == 1)).all().item()
    ratios.append(np.bincount(y.cpu().numpy(), minlength=7) / n)
assert expected["tr"] + expected["val"] == 464_809
print("\nPhân bố nhãn (%), thứ tự lớp 0..6:")
for name, ratio in zip(expected, ratios):
    print(name, np.round(ratio * 100, 4))
np.testing.assert_allclose(ratios[0], ratios[1], atol=1e-4, rtol=0)
np.testing.assert_allclose(ratios[0], ratios[2], atol=1e-4, rtol=0)

# Đối chiếu dữ liệu gốc để kiểm tra cột nhị phân và thống kê chỉ lấy từ train.
X_full, y_full, X_eval_raw, _, ids = load_split(f"{REPO_ROOT}/data/processed")
X_raw, _, X_val_raw, _ = make_val_split(X_full, y_full, 0.2, 42)
np.testing.assert_allclose(data["mean"], X_raw[:, :10].mean(0, dtype=np.float64))
raw_std = X_raw[:, :10].std(0, dtype=np.float64)
np.testing.assert_allclose(data["std"], np.where(raw_std == 0, 1, raw_std))
for name, raw in (("tr", X_raw), ("val", X_val_raw), ("eval", X_eval_raw)):
    np.testing.assert_array_equal(data[f"X_{name}"].cpu().numpy()[:, 10:], raw[:, 10:])
np.testing.assert_array_equal(data["eval_row_id"], ids)
numeric = data["X_tr"].cpu().numpy()[:, :10]
mean = numeric.mean(0, dtype=np.float64)
std = numeric.std(0, dtype=np.float64)
print("\nMean 10 cột train sau chuẩn hoá:", np.round(mean, 7))
print("Std 10 cột train sau chuẩn hoá:", np.round(std, 7))
np.testing.assert_allclose(mean, 0, atol=1e-5)
np.testing.assert_allclose(std, 1, atol=1e-5)
majority = int(data["y_tr"].bincount(minlength=7).argmax().item())
majority_acc = (data["y_val"] == majority).double().mean().item()
print(f"Majority class = {majority}; val accuracy = {majority_acc:.6f}")

# Xáo bằng generator cùng device; giữ batch cuối, không bỏ mẫu.
g = torch.Generator(device=device).manual_seed(42)
seen, batches, last_batch = 0, 0, 0
for xb, yb in iterate_batches(data["X_tr"], data["y_tr"], 512, generator=g):
    assert len(xb) == len(yb)
    seen += len(xb)
    batches += 1
    last_batch = len(xb)
assert seen == expected["tr"]
print(f"Batch check: {seen} mẫu, {batches} batch, batch cuối {last_batch} mẫu")
print("PASS: tất cả kiểm tra Part 0; eval không tham gia fit chuẩn hoá/chọn cấu hình.")
del X_full, y_full, X_eval_raw, X_raw, X_val_raw, numeric, raw

# Preflight trước mọi artifact Part 1-4.
from part4 import choose_output_dir
OUT_DIR = choose_output_dir(data, OUT_DIR)
print("Verified output directory:", OUT_DIR)


train: X (464809, 54) float32, y (464809,) int64, nhãn 0..6 -> data\processed\train.npz
eval : X (116203, 54) float32, y (116203,) int64, nhãn 0..6 -> data\processed\eval.npz

lớp   train(%)   eval(%)
  0    36.461    36.460
  1    48.760    48.760
  2     6.154     6.154
  3     0.473     0.472
  4     1.634     1.634
  5     2.989     2.989
  6     3.530     3.530

đoán luôn lớp đa số (lớp 1) cho accuracy trên eval = 0.4876

tr: X=(371847, 54), y=(371847,), device=cpu
val: X=(92962, 54), y=(92962,), device=cpu
eval: X=(116203, 54), y=(116203,), device=cpu
Majority class = 1; val accuracy = 0.487597

Phân bố nhãn (%), thứ tự lớp 0..6:
tr [36.4607 48.7598  6.1536  0.473   1.6337  2.9891  3.5299]
val [36.4601 48.7597  6.1541  0.4722  1.634   2.9894  3.5305]
eval [36.4603 48.7604  6.1539  0.4724  1.6342  2.9887  3.53  ]

Mean 10 cột train sau chuẩn hoá: [-0. -0. -0.  0. -0. -0. -0. -0. -0.  0.]
Std 10 cột train sau chuẩn hoá: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Majority class = 1; val accura

### Nhận xét Part 0
- Train/val/eval được kiểm tra kích thước, dtype, phân bố 7 lớp và chuẩn hoá trong output phía trên.
- Mean/std chỉ fit trên 371 847 mẫu train sau khi tách validation. Nếu fit trên val/eval, thông tin của tập kiểm tra ảnh hưởng tiền xử lý và gây rò rỉ dữ liệu. Val/eval chỉ được transform bằng thống kê train.
- 44 cột nhị phân giữ nguyên; batch cuối được dùng đầy đủ. Validation luôn dùng seed 42, độc lập với seed các thí nghiệm sau.
- Tensor được đưa lên CUDA một lần khi CUDA khả dụng; nếu không, chạy CPU và in rõ device. Không dùng DataLoader.


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [3]:
# ===== Part 1: model, shape, step-0 loss, gradients and overfit 20 samples =====
import math
import matplotlib.pyplot as plt
import torch.nn.functional as F

set_seed(42)
original_threads = torch.get_num_threads()
if device == "cpu":
    torch.set_num_threads(min(4, original_threads))
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)]
print("M-base parameters:", count_params(model))
model.eval()
with torch.no_grad():
    h = torch.randn(8, 54, device=device)
    for layer in model.net:
        h = layer(h)
        print(f"{type(layer).__name__}: {tuple(h.shape)}")
    assert h.shape == (8, 7) and torch.isfinite(h).all().item()

    # CE over the entire validation set, before updates; no evaluation data.
    val_total, scaled_total = 0.0, 0.0
    for start in range(0, len(data["y_val"]), 8192):
        xb = data["X_val"][start:start + 8192]
        yb = data["y_val"][start:start + 8192]
        logits = model(xb)
        val_total += F.cross_entropy(logits, yb, reduction="sum").item()
        # GUIDE diagnostic: logits x0.1 equals scaling final W/b by 0.1.
        # Diagnostic only; preserve the original He model.
        scaled_total += F.cross_entropy(logits * 0.1, yb, reduction="sum").item()
    step0_loss = val_total / len(data["y_val"])
    scaled_ce = scaled_total / len(data["y_val"])
    ln7 = math.log(7)
    uniform_ce = F.cross_entropy(torch.zeros(20, 7, device=device), data["y_tr"][:20]).item()
    assert math.isfinite(step0_loss) and abs(uniform_ce - ln7) < 1e-6
    assert abs(scaled_ce - ln7) < 0.1
    stats = activation_stats(model, data["X_val"][:512])
print(f"He step-0 CE (val): {step0_loss:.6f}; ln(7): {ln7:.6f}")
print(f"Diagnostic logits x0.1 CE: {scaled_ce:.6f}; uniform CE: {uniform_ce:.6f}")
print("Activation std after each Linear:", stats)
print("Random He logits are not uniform; CE can exceed ln(7). Scaling logits is a diagnostic only; baseline He remains unchanged.")

# Every parameter tensor must have a finite, nonzero gradient norm.
model.zero_grad(set_to_none=True)
F.cross_entropy(model(data["X_tr"][:20]), data["y_tr"][:20]).backward()
gradient_norms = {}
for name, parameter in model.named_parameters():
    assert parameter.grad is not None and torch.isfinite(parameter.grad).all().item()
    gradient_norms[name] = parameter.grad.norm().item()
    assert gradient_norms[name] > 0
    print(f"Gradient {name}: {gradient_norms[name]:.6f}")
model.zero_grad(set_to_none=True)

# Separate model: dropout/weight decay disabled, only 20 training samples.
set_seed(42)
small_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
xb, yb = data["X_tr"][:20], data["y_tr"][:20]
small_optimizer = torch.optim.Adam(small_model.parameters(), lr=0.01, weight_decay=0.0)
history = {"step": [], "loss": [], "acc": [], "grad_norm": []}
steps = 500
for step in range(steps + 1):
    small_model.eval()
    with torch.no_grad():
        logits = small_model(xb)
        measured_loss = F.cross_entropy(logits, yb).item()
        measured_acc = (logits.argmax(1) == yb).float().mean().item()
    assert math.isfinite(measured_loss)
    history["step"].append(step)
    history["loss"].append(measured_loss)
    history["acc"].append(measured_acc)
    if step == steps:
        break
    small_model.train()
    small_optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(small_model(xb), yb)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(small_model.parameters(), float("inf")).item()
    assert math.isfinite(grad_norm)
    history["grad_norm"].append(grad_norm)
    small_optimizer.step()
assert measured_loss < 0.01 and measured_acc == 1.0
print(f"Overfit 20 train samples: {steps} steps, loss={measured_loss:.8f}, accuracy={measured_acc:.1%}")

figure_file = "figures/part1_overfit20.png"
Path(OUT_DIR, "figures").mkdir(parents=True, exist_ok=True)
Path(OUT_DIR, "results").mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].semilogy(history["step"], history["loss"])
axes[0].set_ylabel("Cross-entropy (log scale)")
axes[1].plot(history["step"], history["acc"])
axes[1].set_ylabel("Accuracy")
axes[2].semilogy(history["step"][:-1], history["grad_norm"])
axes[2].set_ylabel("Global gradient norm (no clipping)")
for ax in axes:
    ax.set_xlabel("Update step")
    ax.grid(alpha=0.3)
fig.suptitle("Part 1: overfit 20 train samples | M-base, He, Adam lr=0.01, seed=42")
fig.tight_layout()
fig.savefig(Path(OUT_DIR, figure_file), dpi=150)
plt.close(fig)
part1_summary = dict(params=count_params(model), step0_loss=step0_loss, ln7=ln7,
                     uniform_ce=uniform_ce, scaled_logits_ce=scaled_ce,
                     activation_std=stats, gradient_norms=gradient_norms, steps=steps,
                     overfit_loss=measured_loss, overfit_acc=measured_acc, device=device,
                     torch_version=torch.__version__, seed=42, figure_file=figure_file)
result = dict(summary=part1_summary, history=history,
              cfg=dict(hidden=[256, 128], init="he", dropout=0.0, optimizer="adam",
                       lr=0.01, weight_decay=0.0, batch=20, seed=42,
                       step0_split="validation", overfit_split="train_first20",
                       diagnostic_logits_scale=0.1))
Path(OUT_DIR, "results/part1_checks.json").write_text(
    json.dumps(result, indent=2, allow_nan=False) + "\n", encoding="utf-8")
torch.set_num_threads(original_threads)
print("PASS: Part 1, including diagnosis of He loss deviation from ln(7).")


M-base parameters: 47879
Linear: (8, 256)
ReLU: (8, 256)
Dropout: (8, 256)
Linear: (8, 128)
ReLU: (8, 128)
Dropout: (8, 128)
Linear: (8, 7)
He step-0 CE (val): 2.377572; ln(7): 1.945910
Diagnostic logits x0.1 CE: 1.970439; uniform CE: 1.945910
Activation std after each Linear: [0.6961539387702942, 0.6613748073577881, 0.6523211598396301]
Random He logits are not uniform; CE can exceed ln(7). Scaling logits is a diagnostic only; baseline He remains unchanged.
Gradient net.0.weight: 1.041260
Gradient net.0.bias: 0.432656
Gradient net.3.weight: 3.087581
Gradient net.3.bias: 0.540316
Gradient net.6.weight: 2.499903
Gradient net.6.bias: 0.649802
Overfit 20 train samples: 500 steps, loss=0.00000076, accuracy=100.0%
PASS: Part 1, including diagnosis of He loss deviation from ln(7).


**Nhận xét Part 1 (kết quả thực đo):**
- M-base có 47,879 tham số; logits `(8,7)`.
- He init: CE trên toàn bộ val trước cập nhật = 2.377572, so với ln(7) = 1.945910. Giá trị He lệch mốc, chưa phải phân phối dự đoán đồng đều.
- Chẩn đoán theo GUIDE: logits ×0.1 cho CE = 1.970439; logits bằng 0 cho CE = 1.945910. Kết quả phù hợp với nguyên nhân thang logits ngẫu nhiên của He. Phép giảm thang chỉ là đối chứng, không sửa khởi tạo He của baseline.
- Model riêng quá khớp 20 mẫu sau 500 bước: loss = 0.000001, accuracy = 100.0%.
- Cả 6 tensor tham số có gradient hữu hạn, norm khác 0 trước huấn luyện. Không dùng eval để kiểm tra model hoặc điều chỉnh.
- Thống kê kích hoạt đo sau mỗi Linear. Model thử 20 mẫu được tách riêng; Part 2 phải khởi tạo baseline mới.

![Quá khớp 20 mẫu](../figures/part1_overfit20.png)


## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [4]:
# ===== Part 2: chọn LR bằng val, baseline mới cho mỗi seed =====
from part2 import run_part2
part2_results, part2_summary = run_part2(data, OUT_DIR)
cfg = {**DEFAULT_CFG, "lr": part2_summary["selected_lr"]}


Resume: lr-0.01-s1 (verified data/code fingerprint)
Metrics-only resume: rerun run_experiment(cfg, data) before future final_eval.
Resume: lr-0.03-s1 (verified data/code fingerprint)
Metrics-only resume: rerun run_experiment(cfg, data) before future final_eval.
Resume: lr-0.1-s1 (verified data/code fingerprint)
Metrics-only resume: rerun run_experiment(cfg, data) before future final_eval.
LR screening (selection = lowest validation loss, not eval):
0.01 {'step0_loss': 2.269062042236328, 'best_val_loss': 0.49523329734802246, 'best_epoch': 3, 'final_train_loss': 0.4961846172809601, 'final_val_loss': 0.49523329734802246, 'val_acc': 0.7915384780878209, 'val_macro_f1': 0.598558045297308, 'time_per_epoch_s': 3.6781872999999905, 'peak_mem_MB': None, 'diverged': False, 'initial_activation_std': [0.6662053465843201, 0.6463871598243713, 0.5933443903923035], 'clipped_steps': 0, 'total_steps': 2181, 'clip_fraction': 0.0}
0.03 {'step0_loss': 2.269062042236328, 'best_val_loss': 0.4325620234012604, '

**Nhận xét baseline (kết quả thực đo Part 2):**
- Sàng lọc LR 0.01 / 0.03 / 0.1 cùng seed 1 và 3 epoch; chọn LR = 0.1 theo val loss thấp nhất. Budget ngắn chỉ dùng sàng lọc, không khẳng định LR tối ưu toàn cục.
- Baseline mới: M-base, He, CE, SGD momentum 0.9, batch 512, FP32, dropout/weight decay = 0, không clip; 20 epoch × 3 seed. Split giữ seed 42; train loss đo eval trên 50,000 mẫu cố định.
- base-s1: step0 CE=2.269062; best epoch=19, val loss=0.238200, accuracy=90.35%, macro-F1=0.851984. Train loss 0.4858 → 0.2168; val loss 0.4834 → 0.2408; gap cuối=0.0240.
  Val loss cuối cao hơn best: có dao động/giảm chất lượng sau best; cần đọc cả đường cong, chưa kết luận quá khớp chỉ từ một epoch.
- base-s2: step0 CE=1.978157; best epoch=20, val loss=0.224481, accuracy=91.01%, macro-F1=0.856881. Train loss 0.4819 → 0.2047; val loss 0.4853 → 0.2245; gap cuối=0.0198.
  Val loss cuối đạt best ở epoch cuối: còn cải thiện trong budget; cần đọc cả đường cong, chưa kết luận quá khớp chỉ từ một epoch.
- base-s3: step0 CE=1.900540; best epoch=19, val loss=0.230482, accuracy=90.88%, macro-F1=0.855648. Train loss 0.4664 → 0.2180; val loss 0.4651 → 0.2421; gap cuối=0.0242.
  Val loss cuối cao hơn best: có dao động/giảm chất lượng sau best; cần đọc cả đường cong, chưa kết luận quá khớp chỉ từ một epoch.
- val_acc: mean ± sample std = 0.907457 ± 0.003517; mốc nhiễu 2σ = 0.007034. Chỉ 3 seed, không phải kiểm định thống kê.
- val_macro_f1: mean ± sample std = 0.854838 ± 0.002547; mốc nhiễu 2σ = 0.005093. Chỉ 3 seed, không phải kiểm định thống kê.
- best_val_loss: mean ± sample std = 0.231054 ± 0.006877; mốc nhiễu 2σ = 0.013754. Chỉ 3 seed, không phải kiểm định thống kê.
- Mốc đoán đa số val=48.76%; cả best và epoch cuối của 3 baseline đều vượt mốc: True.
- Step0 không ép về ln(7)=1.945910: He có logits ngẫu nhiên khác 0, như chẩn đoán Part 1. Giữ đúng He của đề.
- Chạy CPU; peak GPU memory không áp dụng (null). Không dùng eval để chọn LR hoặc epoch. JSON và PNG lưu sau mỗi run.

![Baseline seed 1](../figures/base-s1.png)

![So sánh seed](../figures/compare_baseline_seeds.png)


## Part 3 — Thí nghiệm có kiểm soát

Cùng split seed 42, seed 1 và budget 20 epoch cho ablation. SGD momentum và Adam đều thử ít nhất 2 LR.
Cặp high-LR chỉ khác clipping; cấu hình cuối tăng lên 40 epoch × 3 seed, ghi rõ budget khác.
Dự đoán lưu trước chạy vào results/part3_plan.json. Không dùng eval khi chọn cấu hình.


### Dự đoán trước từng thí nghiệm

- **opt-sgdm-0.03** (optimizer): SGD momentum LR 0.03 may converge more slowly than 0.1. Momentum accumulates gradients; LR controls the size of each update.
- **opt-adam-0.001** (optimizer): Adam may converge faster through per-parameter adaptive updates. Adam uses bias-corrected first/second moments; optimizer and its LR change together for fair tuning.
- **opt-adam-0.003** (optimizer): Larger Adam LR may accelerate early fitting but increase oscillation. Compare each optimizer at its own best validation LR with the same 20-epoch budget.
- **loss-mse** (loss): MSE on raw logits versus one-hot targets may fit classification less effectively than CE. MSE averages (logit-one_hot)^2 over B x 7 without 1/2; it is not MSE on softmax probabilities. CE gradient is p-y; logit MSE gradient is 2*(z-y)/7. Compare F1, not loss scales.
- **batch-2048** (hparam): Larger batch reduces updates per epoch and may slow convergence at fixed LR. Batch 2048 changes update count, not just memory: ceil(N/2048) vs ceil(N/512) per epoch.
- **drop-0.1** (dropout): Dropout may reduce the train-val gap but hurt a baseline still improving at epoch 20. Dropout adds training noise and reduces co-adaptation; train loss is measured in eval mode.
- **clip-normal** (clipping): Clipping should activate at normal LR and may slow learning. Global norm threshold is half the baseline epoch-mean norm; actual clipped-step fraction is recorded.
- **clip-high-none** (clipping): Ten-fold LR may cause large gradients, unstable loss, or divergence. Stress-test reference changes LR; compare clipping only against this same high-LR run.
- **clip-high-on** (clipping): At the same high LR clipping may limit spikes, but cannot guarantee stable convergence. Compared with clip-high-none only clip_norm changes; norms are logged before clipping.
- **amp-bf16** (amp): Mixed precision may not improve end-to-end speed on this small MLP; numerical trajectories may change. BF16 has an FP32-like exponent range; FP16 has narrower range and needs loss scaling. Parameters and evaluation remain FP32.
- **init-zeros** (init): Zero initialization leaves hidden ReLU gradients zero; only output bias can learn class priors. Identical zero neurons preserve symmetry and ReLU derivative at zero is zero.
- **init-normal** (init): Small normal weights shrink activations and may delay convergence. Normal std=0.01 is independent of fan-in; repeated layers can attenuate signal.
- **init-xavier** (init): Xavier may shrink ReLU activations relative to He; a shallow MLP may still train well. Xavier normal variance=2/(fan_in+fan_out); He variance=2/fan_in for ReLU.


In [5]:
# ===== Part 3: ablation, validation selection, final three seeds =====
from part3 import run_part3
part3_results, part3_summary = run_part3(data, OUT_DIR, part2_results, part2_summary)


Resume verified: opt-sgdm-0.03
Resume verified: opt-adam-0.001
Resume verified: opt-adam-0.003
Resume verified: loss-mse
Resume verified: batch-2048
Resume verified: drop-0.1
Resume verified: clip-normal
Resume verified: clip-high-none
Resume verified: clip-high-on
Resume verified: amp-bf16
Resume verified: init-zeros
Resume verified: init-normal
Resume verified: init-xavier
Resume verified: final-s1
Resume verified: final-s2
Resume verified: final-s3
{
  "candidate_exp_id": "opt-adam-0.003",
  "final_exp_id": "final-s3",
  "selection": "maximum validation macro-F1 at minimum-loss checkpoint; accuracy tie-break",
  "extended_budget": 40,
  "baseline_noise_2sigma": 0.00509345291162492,
  "eval_used": false,
  "clip_threshold": 0.287,
  "provenance": {
    "fingerprint": "d2e3ad916bfe1dc56a9b137f941509e929e86309b6f31939cc17b9ba8b3fb261",
    "torch": "2.8.0+cu128",
    "device": "cpu",
    "platform": "Windows-11-10.0.26200-SP0",
    "threads": 2,
    "train_rows": 371847,
    "val_rows"

### Đối chiếu sau khi chạy


Dự đoán dưới đây được ghi vào `results/part3_plan.json` trước khi chạy. Mỗi run có JSON, dòng Excel và ảnh riêng. Metric tại checkpoint có val loss thấp nhất; so CE/MSE bằng F1, không so loss khác thang. Đối chiếu từ một seed chỉ mô tả hướng quan sát, chưa chứng minh cơ chế nhân quả.

### Hàm mất mát

- [`loss-mse`](../figures/loss-mse.png): dự đoán **MSE on raw logits versus one-hot targets may fit classification less effectively than CE.** Thực đo: F1=0.7360664509053169, best epoch=19, diverged=False; ΔF1 so `base-s1`=-0.115918, vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (ΔF1 so `base-s1`=-0.115918, vượt 2σ baseline). MSE averages (logit-one_hot)^2 over B x 7 without 1/2; it is not MSE on softmax probabilities. CE gradient is p-y; logit MSE gradient is 2*(z-y)/7. Compare F1, not loss scales.

[Ảnh chồng loss](../figures/compare_loss.png).

### Bộ tối ưu

- [`opt-sgdm-0.03`](../figures/opt-sgdm-0.03.png): dự đoán **SGD momentum LR 0.03 may converge more slowly than 0.1.** Thực đo: F1=0.8264273145468909, best epoch=20, diverged=False; ΔF1 so `base-s1`=-0.025557, vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (ΔF1 so `base-s1`=-0.025557, vượt 2σ baseline). Momentum accumulates gradients; LR controls the size of each update.

- [`opt-adam-0.001`](../figures/opt-adam-0.001.png): dự đoán **Adam may converge faster through per-parameter adaptive updates.** Thực đo: F1=0.8458527882996129, best epoch=18, diverged=False; ΔF1 so `base-s1`=-0.006132, vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (F1 epoch 3=0.711201 vs baseline=0.706247). Adam uses bias-corrected first/second moments; optimizer and its LR change together for fair tuning.

- [`opt-adam-0.003`](../figures/opt-adam-0.003.png): dự đoán **Larger Adam LR may accelerate early fitting but increase oscillation.** Thực đo: F1=0.8740348173262795, best epoch=20, diverged=False; ΔF1 so `base-s1`=+0.022050, vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (F1 epoch 3=0.754287 vs baseline=0.706247). Compare each optimizer at its own best validation LR with the same 20-epoch budget.

[Ảnh chồng optimizer](../figures/compare_optimizer.png).

So ở LR tốt nhất **trong các LR đã thử**, cùng budget 20 epoch:

| Optimizer | exp_id | LR | val F1 | best epoch |
|---|---|---:|---:|---:|
| sgd_momentum | base-s1 | 0.1 | 0.851984 | 19 |
| adam | opt-adam-0.003 | 0.003 | 0.874035 | 20 |


AdamW không được thử; với weight_decay=0, Adam và AdamW có cùng cập nhật. Khi decay>0, AdamW tách decay khỏi moments.

### Batch size

- [`batch-2048`](../figures/batch-2048.png): dự đoán **Larger batch reduces updates per epoch and may slow convergence at fixed LR.** Thực đo: F1=0.8094982220859012, best epoch=20, diverged=False; ΔF1 so `base-s1`=-0.042486, vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (ΔF1 so `base-s1`=-0.042486, vượt 2σ baseline). Batch 2048 changes update count, not just memory: ceil(N/2048) vs ceil(N/512) per epoch.

[Ảnh chồng hparam](../figures/compare_hparam.png).

371,847 mẫu: batch 512 có 727 updates/epoch; batch 2048 có 182. Thời gian epoch lần lượt 3.191s / 2.156s. Không thử scaling LR/warmup; chưa thể kết luận batch lớn kèm LR tối ưu.

### Dropout

- [`drop-0.1`](../figures/drop-0.1.png): dự đoán **Dropout may reduce the train-val gap but hurt a baseline still improving at epoch 20.** Thực đo: F1=0.8417663954300617, best epoch=20, diverged=False; ΔF1 so `base-s1`=-0.010218, vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (ΔF1 so `base-s1`=-0.010218, vượt 2σ baseline). Dropout adds training noise and reduces co-adaptation; train loss is measured in eval mode.

[Ảnh chồng dropout](../figures/compare_dropout.png).

Gap cuối baseline=0.024025, dropout=0.008703. Khoảng cách nhỏ và baseline còn cải thiện không chứng minh quá khớp; dropout chỉ nên dùng khi có evidence train giảm nhưng val tăng, không mặc định giúp.

### Gradient clipping

- [`clip-normal`](../figures/clip-normal.png): dự đoán **Clipping should activate at normal LR and may slow learning.** Thực đo: F1=0.8277551614149751, best epoch=18, diverged=False; ΔF1 so `base-s1`=-0.024229, vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (ΔF1 so `base-s1`=-0.024229, vượt 2σ baseline). Global norm threshold is half the baseline epoch-mean norm; actual clipped-step fraction is recorded.

- [`clip-high-none`](../figures/clip-high-none.png): dự đoán **Ten-fold LR may cause large gradients, unstable loss, or divergence.** Thực đo: F1=0.744103453992362, best epoch=20, diverged=False; ΔF1 so `base-s1`=-0.107881, vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (4 lần val loss tăng >5%; diverged=False). Stress-test reference changes LR; compare clipping only against this same high-LR run.

- [`clip-high-on`](../figures/clip-high-on.png): dự đoán **At the same high LR clipping may limit spikes, but cannot guarantee stable convergence.** Thực đo: F1=0.8090982415774659, best epoch=19, diverged=False; ΔF1 so `base-s1`=-0.042886, vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (high-LR no-clip F1=0.744103453992362, clipped F1=0.8090982415774659). Compared with clip-high-none only clip_norm changes; norms are logged before clipping.

[Ảnh chồng clipping](../figures/compare_clipping.png).

`clip-normal`: clip=0.287, 14538/14540 bước bị cắt (99.99%), diverged=False.

`clip-high-none`: clip=None, 0/14540 bước bị cắt (0.00%), diverged=False.

`clip-high-on`: clip=0.287, 1986/14540 bước bị cắt (13.66%), diverged=False.

Cặp LR=1.0 chỉ khác clip; clipping giới hạn norm gradient, không giới hạn trực tiếp bước momentum hoặc khôi phục neuron ReLU đã chết. Không mặc định gọi đó là “cứu được”.

### Mixed precision

- [`amp-bf16`](../figures/amp-bf16.png): dự đoán **Mixed precision may not improve end-to-end speed on this small MLP; numerical trajectories may change.** Thực đo: F1=0.8500407518872137, best epoch=18, diverged=False; ΔF1 so `base-s1`=-0.001944, chưa vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (median time/epoch=65.670s vs FP32=3.155s). BF16 has an FP32-like exponent range; FP16 has narrower range and needs loss scaling. Parameters and evaluation remain FP32.

[Ảnh chồng amp](../figures/compare_amp.png).

Wall-clock mean: FP32 3.191s/epoch; BF16 2232.532s/epoch. Median: FP32=3.155s, BF16=65.670s, ratio=0.048×; BF16 không nhanh hơn theo median quan sát. Epoch outlier >5×median=[8, 12]; mean có thể chứa quãng gián đoạn thực thi, không diễn giải thành chi phí tính toán. GPU memory=null trên CPU. FP16: unavailable: FP16 training requires CUDA; measured BF16 on CPU; BF16: CPU autocast measured. Không suy rộng CPU sang GPU hoặc coi median là benchmark kiểm soát nhiễu; timing gồm đánh giá FP32. FP16 exponent hẹp cần GradScaler; BF16 exponent 8 bit như FP32 thường không cần scaling.

### Khởi tạo

- [`init-zeros`](../figures/init-zeros.png): dự đoán **Zero initialization leaves hidden ReLU gradients zero; only output bias can learn class priors.** Thực đo: F1=0.09364999018625456, best epoch=14, diverged=False; ΔF1 so `base-s1`=-0.758334, vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (ΔF1 so `base-s1`=-0.758334, vượt 2σ baseline). Identical zero neurons preserve symmetry and ReLU derivative at zero is zero.

- [`init-normal`](../figures/init-normal.png): dự đoán **Small normal weights shrink activations and may delay convergence.** Thực đo: F1=0.8492641904306735, best epoch=20, diverged=False; ΔF1 so `base-s1`=-0.002720, chưa vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (ΔF1 so `base-s1`=-0.002720, chưa vượt 2σ baseline). Normal std=0.01 is independent of fan-in; repeated layers can attenuate signal.

- [`init-xavier`](../figures/init-xavier.png): dự đoán **Xavier may shrink ReLU activations relative to He; a shallow MLP may still train well.** Thực đo: F1=0.8469795188443587, best epoch=18, diverged=False; ΔF1 so `base-s1`=-0.005005, chưa vượt 2σ baseline. Đối chiếu: phù hợp hướng dự đoán (std kích hoạt so với He trong bảng bên dưới). Xavier normal variance=2/(fan_in+fan_out); He variance=2/fan_in for ReLU.

[Ảnh chồng init](../figures/compare_init.png).


| init | step0 loss | Std sau từng Linear, 512 mẫu val |
|---|---:|---|
| he | 2.269062 | 0.666205, 0.646387, 0.593344 |
| zeros | 1.945910 | 0.000000, 0.000000, 0.000000 |
| normal | 1.945996 | 0.034617, 0.003800, 0.000279 |
| xavier | 2.022177 | 0.278051, 0.220274, 0.196886 |


Mạng chỉ có hai lớp ẩn; khác biệt này không chứng minh hành vi mạng 30 lớp trong slide. Zero chỉ học bias lớp ra, không học đặc trưng ẩn.


![So sánh optimizer](../figures/compare_optimizer.png)



In [6]:
# Đối chiếu trực tiếp từ kết quả runtime này.
base = next(r for r in part2_results if r["cfg"]["exp_id"] == "base-s1")
noise = part2_summary["val_macro_f1"]["noise_2sigma"]
print("Baseline macro-F1 noise 2σ:", noise)
for result in part3_results:
    s, c = result["summary"], result["cfg"]
    delta = s["val_macro_f1"] - base["summary"]["val_macro_f1"] if s["val_macro_f1"] is not None else None
    print(c["exp_id"], "F1=", s["val_macro_f1"], "ΔF1=", delta,
          "beyond baseline noise=", abs(delta) > noise if delta is not None else None,
          "best epoch=", s["best_epoch"], "diverged=", s["diverged"])
print("Candidate selected on validation:", part3_summary["candidate_exp_id"])
print("Final seed selected on validation:", part3_summary["final_exp_id"])
print("Final validation F1 mean ± sample std:", part3_summary["final_f1_mean"], part3_summary["final_f1_std"])
print("Comparisons:", sorted(p.name for p in Path(OUT_DIR, "figures").glob("compare_*.png")))


Baseline macro-F1 noise 2σ: 0.00509345291162492
opt-sgdm-0.03 F1= 0.8264273145468909 ΔF1= -0.025557054464925066 beyond baseline noise= True best epoch= 20 diverged= False
opt-adam-0.001 F1= 0.8458527882996129 ΔF1= -0.0061315807122031 beyond baseline noise= True best epoch= 18 diverged= False
opt-adam-0.003 F1= 0.8740348173262795 ΔF1= 0.022050448314463567 beyond baseline noise= True best epoch= 20 diverged= False
loss-mse F1= 0.7360664509053169 ΔF1= -0.1159179181064991 beyond baseline noise= True best epoch= 19 diverged= False
batch-2048 F1= 0.8094982220859012 ΔF1= -0.04248614692591479 beyond baseline noise= True best epoch= 20 diverged= False
drop-0.1 F1= 0.8417663954300617 ΔF1= -0.010217973581754247 beyond baseline noise= True best epoch= 20 diverged= False
clip-normal F1= 0.8277551614149751 ΔF1= -0.024229207596840885 beyond baseline noise= True best epoch= 18 diverged= False
clip-high-none F1= 0.744103453992362 ΔF1= -0.10788091501945396 beyond baseline noise= True best epoch= 20 dive

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [7]:
# ===== Part 4: freeze selection, official evaluation, workbook and report =====
from part4 import run_part4
part4_summary = run_part4(data, OUT_DIR, REPO_ROOT, part2_results, part2_summary,
                          part3_results, part3_summary)


Reusing sealed official evaluation: baseline
Reusing sealed official evaluation: final
Reusing locked workbook: all cell values and formulas match C:\Su\VinLab\phase2-track4\Lec1-Noron\K4-Track4-Day1-Neuralnetwork\submission_2A202602853\experiments.xlsx
{
  "final_exp_id": "final-s3",
  "final_eval_macro_f1": 0.8968796913262628,
  "final_eval_accuracy": 0.9282290474428371,
  "baseline_eval": {
    "n_eval": 116203,
    "accuracy": 0.9016118344621051,
    "macro_f1": 0.8514340846236169,
    "per_class": [
      {
        "cls": 0,
        "support": 42368,
        "precision": 0.9037180313776059,
        "recall": 0.8932449018126888,
        "f1": 0.8984509466437178
      },
      {
        "cls": 1,
        "support": 56661,
        "precision": 0.9128656095386283,
        "recall": 0.9215333298035686,
        "f1": 0.9171789917442473
      },
      {
        "cls": 2,
        "support": 7151,
        "precision": 0.8910560044580663,
        "recall": 0.8944203607887009,
        "f1": 

In [8]:
# Phân tích lỗi từ kết quả script chính thức.
scores = json.loads(Path(OUT_DIR, "eval_result.json").read_text(encoding="utf-8"))
print("Rows:", scores["n_eval"], "accuracy:", scores["accuracy"], "macro-F1:", scores["macro_f1"])
for row in scores["per_class"]:
    print(row)
hardest = min(scores["per_class"], key=lambda row: row["f1"])
errors = np.array(scores["confusion_matrix"])[hardest["cls"]].copy()
errors[hardest["cls"]] = 0
print("Hardest class:", hardest["cls"], "most confused with:", int(errors.argmax()))
print("Class imbalance is a possible mechanism; feature similarity has not been measured.")
print("Next experiment: class-weighted CE selected on validation.")


Rows: 116203 accuracy: 0.9282290474428371 macro-F1: 0.8968796913262628
{'cls': 0, 'support': 42368, 'precision': 0.9388727370452713, 'recall': 0.9070288897280967, 'f1': 0.9226761425707392}
{'cls': 1, 'support': 56661, 'precision': 0.9269739217135686, 'recall': 0.9516775206932458, 'f1': 0.9391632994287308}
{'cls': 2, 'support': 7151, 'precision': 0.9295655841417123, 'recall': 0.924625926443854, 'f1': 0.9270891755468311}
{'cls': 3, 'support': 549, 'precision': 0.8485401459854015, 'recall': 0.8469945355191257, 'f1': 0.8477666362807658}
{'cls': 4, 'support': 1899, 'precision': 0.8556588299617277, 'recall': 0.8241179568193786, 'f1': 0.8395922746781115}
{'cls': 5, 'support': 3473, 'precision': 0.8735971647962197, 'recall': 0.8517132162395623, 'f1': 0.8625164018078437}
{'cls': 6, 'support': 4102, 'precision': 0.925680473372781, 'recall': 0.9534373476352999, 'f1': 0.9393539089708177}
Hardest class: 4 most confused with: 1
Class imbalance is a possible mechanism; feature similarity has not been

### Phân tích lỗi và đánh giá cuối


Khóa lựa chọn **trước eval**: ứng viên `opt-adam-0.003` thắng theo val F1; giữ hyperparameters và tăng budget lên 40 epoch × 3 seed. Final val F1 mean±sample std=0.894095±0.004972; chọn một mô hình `final-s3` theo val F1, không ensemble. Cấu hình đầy đủ trong `results/final_selection.json`. [Ảnh final](../figures/compare_final.png).


| Cấu hình | Seed | val F1 | eval F1 | eval acc |
|---|---:|---:|---:|---:|
| base-s1 | 1 | 0.851984 | 0.851434 | 0.901612 |
| final-s3 | 3 | 0.898530 | 0.896880 | 0.928229 |


Điểm do `scripts/evaluate.py` tạo: 116,203 row_id hợp lệ. Δeval F1=+0.045446; mốc 2σ val=0.005093, không có nhiều seed eval nên không gọi đây là kiểm định nhiễu eval. Final eval−val F1=-0.001651. Không chỉnh cấu hình sau eval.

### Phân tích lỗi theo lớp

| Lớp (0-based) | Support | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| 0 | 42368 | 0.938873 | 0.907029 | 0.922676 |
| 1 | 56661 | 0.926974 | 0.951678 | 0.939163 |
| 2 | 7151 | 0.929566 | 0.924626 | 0.927089 |
| 3 | 549 | 0.848540 | 0.846995 | 0.847767 |
| 4 | 1899 | 0.855659 | 0.824118 | 0.839592 |
| 5 | 3473 | 0.873597 | 0.851713 | 0.862516 |
| 6 | 4102 | 0.925680 | 0.953437 | 0.939354 |


Lớp khó nhất=4, F1=0.839592, support=1899; nhầm nhiều nhất sang lớp 1: 278 mẫu. [Ma trận nhầm lẫn](../figures/confusion_eval.png), hàng=thật/cột=dự đoán. Dữ liệu mất cân bằng là cơ chế khả dĩ; sự giống nhau đặc trưng chỉ là giả thuyết, chưa phân tích feature để chứng minh. Có thể thử class-weighted CE và đánh giá trên val trong nghiên cứu tiếp theo.


![Ma trận nhầm lẫn](../figures/confusion_eval.png)



In [9]:
# Kiểm tra artifact được tạo bởi run_part4.
required = ["REPORT.md", "experiments.xlsx", "predictions_eval.csv", "eval_result.json"]
assert all(Path(OUT_DIR, name).is_file() for name in required)
results = load_results(str(Path(OUT_DIR, "results")))
assert len(results) == part4_summary["experiment_count"]
assert all(Path(OUT_DIR, "figures", r["cfg"]["exp_id"] + ".png").is_file() for r in results)
print("Complete runs and individual plots:", len(results))
print("Submission artifacts:", required)
print("REPORT:", str(Path(OUT_DIR, "REPORT.md")))
print("Excel formulas recalculate when opened; packaging verifier checks saved caches.")


Complete runs and individual plots: 22
Submission artifacts: ['REPORT.md', 'experiments.xlsx', 'predictions_eval.csv', 'eval_result.json']
REPORT: C:\Su\VinLab\phase2-track4\Lec1-Noron\K4-Track4-Day1-Neuralnetwork\submission_2A202602853\REPORT.md
Excel formulas recalculate when opened; packaging verifier checks saved caches.
